# NóiLái — API runs (Gemini and Groq free tiers), core set only

Runs the API lines of `configs/run_plan.yaml` (`E1_api_core`, `E1_api_paraphrase`,
`E3_api_core`, `reasoning_500_api`) on the four API-served models. Guards in
`scripts/kaggle_run_plan.py` make every API command `--in-core-only` and refuse the sealed
split. Keys come from the environment / Colab userdata / Kaggle Secrets (`GEMINI_API_KEY`,
`GROQ_API_KEY`) and are never printed.

**Daily-quota loop.** Groq's free plan allows 1,000 requests and 200K tokens a day per model,
so the 1,500-item core takes about four days per model. The driver keeps
`data/runs/api_ledger.json` (calls per model per UTC day, counted from new rows in
`outputs.jsonl`) and skips a model whose daily request budget is spent; run this notebook
once a day with `--resume` until every run is complete. Gemini's free-tier limits are shown
only in AI Studio: read them and set `GEMINI_RPD_OVERRIDE` if the run must stop earlier.

Terms: Gemini's free tier trains on inputs and requires users to be 18+; only the core reaches
an API, never annotator data, never the sealed split (plan §2.3, §3).

In [ ]:
# ---- parameters: edit this cell only; nothing below asks for input ------------------------------
import time
SESSION_T0 = time.time()

REPO_URL = "https://github.com/OWNER/REPO.git"     # [UNCERTAIN: verify]
REPO_REF = "main"
REPO_SUBDIR = "noilai"
BUNDLE_PATH = "/content/drive/MyDrive/noilai/noilai-main.bundle"   # Colab; on Kaggle use /kaggle/input/noilai-bundle/noilai-main.bundle
CLONE_DIR = "/content/repo"
WORK_DIR = "/content"
ITEMS_DATASET_DIR = "/content/drive/MyDrive/noilai/release"        # the frozen data/release tree; None if the clone has it
VERIFY_ITEM_KEYS = ["noilai_test"]

GITHUB_TOKEN_SECRET = "GITHUB_TOKEN"
API_KEY_SECRETS = ["GEMINI_API_KEY", "GROQ_API_KEY"]   # names only; values stay in the secret store
GEMINI_RPD_OVERRIDE = None       # requests/day read in AI Studio; None = no ledger cap for Gemini [UNCERTAIN: verify]

RUN_IDS = ["E1_api_core", "E1_api_paraphrase", "E3_api_core"]   # add "reasoning_500_api" for the cost curve
MODELS = []                      # subset of the run's models; [] = all four
EXTRA_ARGS = ""
DRY_RUN = False

PLATFORM, GPU_TYPE, N_GPUS = "api", "api", 0
REQUIRE_GPU = False
RUN_LABEL = "+".join(RUN_IDS)
MOUNT_DRIVE = True

In [ ]:
# secrets: environment -> Colab userdata -> Kaggle Secrets. Values go into os.environ for the
# library that needs them and are reported as found / not found. Nothing here prints a value.
import os

def _secret(name):
    val = os.environ.get(name)
    if val:
        return val
    try:
        from google.colab import userdata          # Colab
        try:
            val = userdata.get(name)
        except Exception:
            val = None
        if val:
            return val
    except ImportError:
        pass
    try:
        from kaggle_secrets import UserSecretsClient   # Kaggle
        try:
            val = UserSecretsClient().get_secret(name)
        except Exception:
            val = None
        if val:
            return val
    except ImportError:
        pass
    return None

def _export(name, required=False):
    val = _secret(name)
    if val:
        os.environ[name] = val
        print(f"secret {name}: found ({len(val)} chars, not shown)")
        return True
    print(f"secret {name}: not found")
    if required:
        raise SystemExit(f"{name} is required for this notebook")
    return False

In [ ]:
# Drive (Colab only): bundle, release and the outputs' backup
from pathlib import Path
if MOUNT_DRIVE:
    try:
        from google.colab import drive
        drive.mount("/content/drive")
        print("Drive mounted")
    except ImportError:
        print("not on Colab: no Drive; set BUNDLE_PATH / ITEMS_DATASET_DIR to local paths")

In [ ]:
# (b) clone: from a git bundle in a private dataset / Drive when present (no token needed), else from
# GitHub with a token from the platform's secret store. The token reaches git only through GIT_ASKPASS,
# a two-line helper that echoes an environment variable: it is never in a URL, an argv or this output.
import os, stat, subprocess, sys
from pathlib import Path

clone_dir = Path(CLONE_DIR)
env = dict(os.environ, GIT_TERMINAL_PROMPT="0")
bundle = Path(BUNDLE_PATH) if BUNDLE_PATH else None
if bundle is not None and not bundle.exists():
    print(f"bundle {bundle} not found; falling back to {REPO_URL}")
    bundle = None
if bundle is None:
    if "OWNER/REPO" in REPO_URL:
        raise SystemExit("set REPO_URL (or provide BUNDLE_PATH) in the parameters cell")
    if _export(GITHUB_TOKEN_SECRET):
        askpass = Path.home() / ".noilai_askpass.sh"
        askpass.write_text("#!/bin/sh\necho \"$" + GITHUB_TOKEN_SECRET + "\"\n")
        askpass.chmod(stat.S_IRWXU)
        env["GIT_ASKPASS"] = str(askpass)
source = str(bundle) if bundle else REPO_URL
if (clone_dir / ".git").exists():
    subprocess.run(["git", "-C", str(clone_dir), "fetch", "--tags", source, "+refs/heads/*:refs/remotes/origin/*"], check=True, env=env)
else:
    subprocess.run(["git", "clone", source, str(clone_dir)], check=True, env=env)
for cand in (REPO_REF, f"origin/{REPO_REF}"):
    if subprocess.run(["git", "-C", str(clone_dir), "checkout", "--detach", cand], env=env).returncode == 0:
        break
else:
    raise SystemExit(f"cannot check out {REPO_REF}")
HEAD = subprocess.run(["git", "-C", str(clone_dir), "rev-parse", "HEAD"], check=True, env=env,
                      capture_output=True, text=True).stdout.strip()
PROJECT = clone_dir / REPO_SUBDIR if (clone_dir / REPO_SUBDIR / "pyproject.toml").exists() else clone_dir
os.chdir(PROJECT)
sys.path.insert(0, str(PROJECT / "scripts"))
print("HEAD", HEAD)
print("project", PROJECT)

In [ ]:
# (c) install: the project with its eval extra (API SDKs: openai, google-genai, groq); no inference engine.
# Record pip freeze + versions next to the outputs (checklist C4).
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", ".[eval]"], check=True)
import colab_setup as CS
ENV_RECORD = CS.record_environment(PROJECT / "data" / "runs" / "env")
print("environment recorded at", ENV_RECORD)

In [ ]:
# API keys into the environment (names from configs/models.yaml providers.*.api_key_env); found / not found only
import yaml
cfg = yaml.safe_load(open("configs/models.yaml", encoding="utf-8"))
wanted = sorted({p["api_key_env"] for p in cfg["providers"].values()} & set(API_KEY_SECRETS))
FOUND = {name: _export(name) for name in wanted}
if not any(FOUND.values()):
    raise SystemExit("no API key found: add GEMINI_API_KEY / GROQ_API_KEY to the secret store")
if GEMINI_RPD_OVERRIDE:
    cfg["providers"]["gemini"]["free_tier"]["requests_per_day"] = int(GEMINI_RPD_OVERRIDE)
    yaml.safe_dump(cfg, open("configs/models.yaml", "w", encoding="utf-8"), allow_unicode=True, sort_keys=False)
    print("Gemini requests/day set to", GEMINI_RPD_OVERRIDE, "for this session's ledger (local edit, not committed)")

In [ ]:
# (d) third-party resources (SHA-256 against data/HASHES.json) and the item file(s): hash against
# configs/run_plan.yaml, canary on every row, row count. A failure stops the notebook here.
import shutil, subprocess, sys
from pathlib import Path
r = subprocess.run([sys.executable, "scripts/fetch_resources.py"], check=False)
if r.returncode:
    raise SystemExit("resource fetch or hash check failed (see above)")
if ITEMS_DATASET_DIR:
    src = Path(ITEMS_DATASET_DIR)
    if not src.exists():
        raise SystemExit(f"ITEMS_DATASET_DIR {src} not found: attach the private dataset that holds data/release")
    shutil.copytree(src, PROJECT / "data" / "release", dirs_exist_ok=True)
    print("release copied from", src)
for key in VERIFY_ITEM_KEYS:
    r = subprocess.run([sys.executable, "scripts/kaggle_verify_items.py", "--key", key], check=False)
    if r.returncode:
        raise SystemExit(f"item file {key} failed verification; nothing runs on it")

### RUN CELL — daily API loop
For each run id and model: skip if today's ledger says the daily request budget is spent, else
`run_eval.py ... --in-core-only --resume`. Models whose key is missing are skipped with a
message. Re-run tomorrow; `--resume` continues.

In [ ]:
# RUN CELL: quota-aware loop over RUN_IDS x MODELS through scripts/kaggle_run_plan.py (in-core-only enforced there).
import json, shlex
import kaggle_run_plan as KRP
plan, models_cfg = KRP.load_plan(), KRP.load_models()
RESULTS = []
for run_id in RUN_IDS:
    run = KRP.find_run(plan, run_id)
    names = MODELS or KRP.expand_models(run, plan, models_cfg)
    runnable = []
    for name in names:
        prov = models_cfg["by_name"][name].get("provider")
        key_env = models_cfg["providers"][prov]["api_key_env"] if prov else None
        if key_env and not os.environ.get(key_env):
            print(f"skip {name}: {key_env} not set")
        else:
            runnable.append(name)
    RESULTS += KRP.execute(run_id, models=runnable, platform=PLATFORM, dry_run=DRY_RUN, continue_on_error=True,
                           extra=shlex.split(EXTRA_ARGS), plan=plan, models_cfg=models_cfg)
print(json.dumps([{"run": r["run"], "model": r["model"], "status": r["status"], "new_outputs": r.get("new_outputs"),
                   "budget": r.get("api_budget_today")} for r in RESULTS], indent=1))
ledger = KRP.ledger_load(KRP.ledger_path(plan))
print("ledger (calls per model per UTC day):", json.dumps(ledger, indent=1))

In [ ]:
# keep the outputs: copy data/runs, the ledger and the compute log to Drive (Colab) or the working dir
import shutil
from pathlib import Path
dest = Path("/content/drive/MyDrive/noilai/runs_api") if Path("/content/drive/MyDrive").exists() else Path(WORK_DIR) / "noilai_runs_out"
shutil.copytree(PROJECT / "data" / "runs", dest / "runs", dirs_exist_ok=True)
import compute_log as CL
LOG = PROJECT / "data" / "compute_log.csv"
CL.append_entry(LOG, CL.Entry(date=CL.today(), platform=PLATFORM, gpu_type=GPU_TYPE, n_gpus=N_GPUS,
                              hours=CL.hours_since(SESSION_T0), run_id=f"session-{RUN_LABEL}", purpose="api_runs"))
shutil.copy2(LOG, dest / "compute_log.csv")
print(CL.format_totals(CL.totals(CL.read_entries(LOG))))
print("outputs copied to", dest)